# CV Masterclass 11: Edge Deployment & TensorRT

Welcome to Notebook 11, the final module in the Computer Vision Masterclass.

A 100-layer ResNet trained in PyTorch on an A100 GPU is mathematically flawless. But if you try to run it on a small drone's Raspberry Pi or an Nvidia Jetson Nano, you will get 2 Frames Per Second (FPS) and the battery will drain in 4 minutes.

Producing an AI model is useless if you cannot deploy it to the Edge.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. It addresses the danger of Post-Training Quantization (PTQ):

> *"When converting a PyTorch FP32 model to an optimized INT8 TensorRT engine, you must provide a 'Calibration Dataset' of ~1,000 images before it converts. Why? What exact statistical properties is the compiler searching for in those 1,000 images, and what happens to your YOLO's bounding boxes if you perform Quantization without them?"*

---

## Table of Contents
1. **The Deployment Pipeline:** PyTorch -> ONNX -> TensorRT.
2. **Graph Optimization:** Layer fusion and memory pruning.
3. **FP32 vs INT8:** The physics of Quantization.
4. **Calibration (Kullback-Leibler Divergence):** Squeezing infinity into 256 bins.


## 1. The Deployment Pipeline

`torch.save(model.state_dict())` is strictly for Python. You cannot reliably run Python on Edge devices (it requires a massive interpreter and GIL overhead).

1.  **ONNX (Open Neural Network Exchange):** We export the PyTorch model to an ONNX file. ONNX throws away all your Python classes and `def forward()` loops. It tracks the exact mathematical sequence of tensors and compiles a static C++ computation graph.
2.  **TensorRT (TRT):** ONNX is agnostic. Nvidia TensorRT takes the ONNX graph and physically compiles it specifically for the exact silicon architecture of the target device (e.g., maximizing the exact number of Tensor Cores available on a Jetson Orin).

## 2. Graph Optimization & Layer Fusion

In PyTorch, a standard layer is: `Conv2d -> BatchNorm2d -> ReLU`.

During inference, reading and writing to GPU memory across three separate algorithmic steps is slow. 

TensorRT physically "fuses" these layers into a single mathematical step. It absorbs the Batch Norm scaling directly into the Convolution Weights (math allows this because they are both linear operations), and appends the ReLU threshold to the kernel output. 

**Result:** 1 memory read, 1 memory write. The model becomes twice as fast before we even touch data types.

In [ ]:
import torch
import torch.nn as nn

# -----------------------------------------------------
# IMPLEMENTATION: Mathematical Batch Norm Fusion
# -----------------------------------------------------

def fuse_conv_and_bn(conv, bn):
    """
    Simulates what TensorRT does natively.
    Absorbs BatchNorm parameters directly into the Conv2d weights.
    """
    # 1. Extract the Batch Norm constants
    bn_mean = bn.running_mean
    bn_var = bn.running_var
    bn_eps = bn.eps
    bn_weight = bn.weight # Gamma
    bn_bias = bn.bias     # Beta

    # 2. Extract Conv parameters
    conv_weight = conv.weight
    conv_bias = conv.bias if conv.bias is not None else torch.zeros_like(bn_mean)

    # 3. The Fusion Math
    bn_std = torch.sqrt(bn_var + bn_eps)
    multiplier = bn_weight / bn_std

    # Apply to Conv Weights
    fused_weight = conv_weight * multiplier.view(-1, 1, 1, 1)
    # Apply to Conv Bias
    fused_bias = (conv_bias - bn_mean) * multiplier + bn_bias

    # Create a new, pure Conv layer (Batch Norm is deleted!)
    fused_conv = nn.Conv2d(conv.in_channels, out_channels=conv.out_channels,
                           kernel_size=conv.kernel_size, stride=conv.stride,
                           padding=conv.padding, bias=True)
                           
    fused_conv.weight = nn.Parameter(fused_weight)
    fused_conv.bias = nn.Parameter(fused_bias)
    
    return fused_conv

print("TensorRT Graph Fusion Mathematics defined.")

## 3. FP32 vs INT8 (Quantization Physics)

PyTorch models use 32-bit Floating Point numbers (`FP32`). These represent massive ranges with infinite decimals.

Edge hardware runs incredibly fast on 8-bit Integers (`INT8`). An `INT8` can only represent exactly 256 physical distinct numbers (e.g., $-128$ to $+127$).

If a Convolution outputs the activation value `3.14159`, how do we map that into the `[-128, +127]` integer bin? We scale it.

## 4. Calibration & Kullback-Leibler Divergence

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does TensorRT demand a "Calibration Dataset" before compiling INT8?*

**A:** 
To map an `FP32` space into exactly 256 an `INT8` bins, TensorRT needs to know the physical maximum value the network outputs. 
If the max activation is `1.0`, it maps `1.0 -> 127`.
If the max activation is `100.0`, it maps `100.0 -> 127`.

But Neural Networks have **Outliers**. Say 99% of your activations are between `-2.0` and `2.0`, but occasionally one neuron screams `90.0`.

If TensorRT naively sets the boundary at `90.0 -> 127`, then the entire critical range of `[-2.0, 2.0]` (where the actual object features live) is compressed into a single INT8 bin (e.g., `Bin 0`). All mathematical detail is utterly destroyed. Your bounding boxes will randomize or disappear.

**The Calibration Solution:**
You feed TensorRT 1,000 images representing the real-world factory. It physically tracks the exact statistical distribution of every single neuron.
It uses **Kullback-Leibler (KL) Divergence** to find the optimal "Clip Value". 

It might decide to set the map boundary at `3.0 -> 127`, preserving the vast majority of the data beautifully, and happily truncating the `90.0` outlier down to `127` as an acceptable loss. 

Without Calibration, INT8 object detection is mathematically impossible.

---
# Computer Vision: Masterclass Complete 🎓

You have successfully navigated from the physics of discrete $3\times3$ convolutions, through the identity mechanics of ResNet, the geometric scaling of U-Net, the infinite dimensional spaces of Vision Transformers, the contrastive dynamics of Self-Supervised Learning, the generative realities of Diffusion models, and finally, compiled it all onto physical Edge hardware.

The 11-module CV theoretical foundation is mathematically secure.